# Multitemporal Analysis & Post-Classification Consistency 

This notebook runs **live Google Earth Engine** integration checks for each phase of the multitemporal plan

## 0. Setup & Earth Engine Initialization

In [ ]:
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception:
    pass

import os
import sys
import ee
import geemap
import pandas as pd
import geopandas as gpd

# Ensure local src/ is importable when running from notebooks/
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
src_path = os.path.join(repo_root, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import luma_ge

# Auto-initialize via auth/*.json (or fall back to manual auth)
ok = luma_ge.auto_initialize()
if not ok:
    luma_ge.authenticate_manually()

status = luma_ge.get_auth_status()
print('Auth status:', status)
assert status['initialized'], 'Earth Engine must be initialized before running live tests!'

In [ ]:
#Define Area of Interest (AOI)
#Kambera, NTT
aoi = ee.FeatureCollection('projects/ee-agilfahrezy60/assets/Sesi1_AOI_Kambera_NTT')
#classification scheme
from luma_ge.classification_scheme import LULC_Scheme_Manager
manager = LULC_Scheme_Manager()
#path to csv 
csv_path = "../data/MODUL2_Skema_Klasifikasi_Kambera.csv"
scheme_df = pd.read_csv(csv_path, sep=None, engine="python")
print("", scheme_df)
#Training data
roi = ee.FeatureCollection('projects/ee-agilfahrezy60/assets/Sesi1_Data_Latih_Kambera') 
#validation data
validation = ee.FeatureCollection('projects/ee-agilfahrezy60/assets/Sesi2_Data_Penguji_Akurasi')

---
## Phase 1 Live EE Tests: `imagery_correction.py` & `data_acquisition.py`
Tests:
1. **1A**: `Reflectance_Data.get_hls_optical_data()` (NASA HLS v2.0 `Fmask` + `0.0001` scaling)
2. **1B**: `BRDF_Correction` + `Topographic_Correction` on a live Landsat 8 scene
3. **1C**: `get_multitemporal_optical_data()` (`L5_SR` 2010 + `L8_SR` 2024 with Roy 2016 + optional BRDF) and `get_multitemporal_composites()` (dropping `AEROSOL` + CDF `histogram_match`)

In [ ]:
# Test 1A: NASA HLS v2.0 Live Acquisition
from luma_ge.data_acquisition import Reflectance_Data, final_Image

rd = Reflectance_Data()
hls_col, hls_stats = rd.get_hls_optical_data(
    aoi=aoi,
    start_date='2025-01-01',
    end_date='2025-12-31',
    dataset='HLS',
    cloud_cover=30,
    verbose=True,
    compute_detailed_stats=True,
)
print('HLS filtered image count:', hls_stats['filtered_collection']['total_images'])
if hls_stats['filtered_collection']['total_images'] > 0:
    print('HLS first image bands:', hls_col.first().bandNames().getInfo())
    hls_comp = final_Image().get_temporal_composite(hls_col, aoi, verbose=False)
    print('HLS composite bands:', hls_comp['image'].bandNames().getInfo())

In [ ]:
# Test 1B: Live BRDF_Correction & Topographic_Correction (SCSc) on Landsat 8 SR
from luma_ge.imagery_correction import BRDF_Correction, Topographic_Correction

l8_col, _ = rd.get_optical_data(
    aoi=aoi,
    start_date='2025-01-01',
    end_date='2025-12-31',
    optical_data='L8_SR',
    cloud_cover=30,
    verbose=False,
    compute_detailed_stats=False,
)
sample_scene = ee.Image(l8_col.first())

brdf = BRDF_Correction()
topo = Topographic_Correction()

scene_brdf = brdf.correct_image(sample_scene, verbose=True)
scene_brdf_topo = topo.correct_image(scene_brdf, scale=30, verbose=True)

# Verify properties and median reflectance on server
raw_med = sample_scene.select(['RED', 'NIR']).reduceRegion(ee.Reducer.median(), aoi, 60).getInfo()
corr_med = scene_brdf_topo.select(['RED', 'NIR']).reduceRegion(ee.Reducer.median(), aoi, 60).getInfo()
print('Raw scene median (RED/NIR):', raw_med)
print('BRDF + Topo corrected median (RED/NIR):', corr_med)
print('system:time_start preserved:', scene_brdf_topo.get('system:time_start').getInfo() is not None)

In [ ]:
# Test 1C: Cross-Sensor Multitemporal Acquisition (2010 L5_SR + 2025 L8_SR) + CDF Histogram Matching
periods_cfg = {
    'baseline': {'start_date': 2025, 'end_date': 2025, 'sensor': 'L8_SR', 'cloud_cover': 30},
    'historical': {'start_date': 2010, 'end_date': 2010, 'sensor': 'L5_SR', 'cloud_cover': 60},
}

mt_data = rd.get_multitemporal_optical_data(
    aoi=aoi,
    periods=periods_cfg,
    harmonize_landsat=True,
    apply_brdf=True,
    apply_topo_correction=False,
    verbose=True,
    compute_detailed_stats=True,
)
print('Chronological order:', mt_data['period_order'])

fi = final_Image()
mt_composites = fi.get_multitemporal_composites(
    collections_by_period=mt_data,
    aoi=aoi,
    reference_period='baseline',
    reducer='median',
    drop_aerosol=True,
    harmonize_sensors=True,
    histogram_match=True,
    scale=30,
    max_buckets=128,
    verbose=True,
)
print('Common bands (AEROSOL dropped):', mt_composites['common_bands'])

# Compare median reflectance before vs after histogram matching
geom = aoi.geometry() if isinstance(aoi, ee.FeatureCollection) else aoi
raw_hist_med = mt_composites['raw_composites']['historical'].select(['BLUE', 'RED', 'NIR']).reduceRegion(ee.Reducer.median(), geom, 60).getInfo()
matched_hist_med = mt_composites['composites']['historical'].select(['BLUE', 'RED', 'NIR']).reduceRegion(ee.Reducer.median(), geom, 60).getInfo()
base_med = mt_composites['composites']['baseline'].select(['BLUE', 'RED', 'NIR']).reduceRegion(ee.Reducer.median(), geom, 60).getInfo()
print('Historical RAW median:', raw_hist_med)
print('Historical MATCHED median:', matched_hist_med)
print('Baseline (reference) median:', base_med)

Map1 = geemap.Map()
vis_rgb = {'bands': ['RED', 'GREEN', 'BLUE'], 'min': 0.0, 'max': 0.25, 'gamma': 1.3}
Map1.addLayer(mt_composites['raw_composites']['historical'], vis_rgb, '2010 Historical (Raw)')
Map1.addLayer(mt_composites['composites']['historical'], vis_rgb, '2010 Historical (Histogram Matched)')
Map1.addLayer(mt_composites['composites']['baseline'], vis_rgb, '2024 Baseline')
Map1.centerObject(aoi, 12)
Map1

---
## Phase 2 Live EE Tests: `sample_data.py` & `sample_data_quality.py`
Tests:
1. **2A**: `SyncTrainData.LoadMultitemporalTrainData` (1 dataset ? Model Transfer routing; train/test split is handled downstream in `FeatureExtraction`)
2. **2B**: `sample_quality.run_multitemporal_separability` (live GEE pixel extraction + TD distance on `baseline` only, skipping `historical`)

In [ ]:
from luma_ge.sample_data import SyncTrainData
from luma_ge.sample_data_quality import sample_quality

# Kambera EE FeatureCollection uses 'ID_Kelas' (class ID) and 'KelasTupla' (class name)
CLASS_ID_PROP = 'ID_Kelas'
CLASS_NAME_PROP = 'KelasTupla'

train_loaded = SyncTrainData.LoadMultitemporalTrainData(
    training_sources={'historical': None, 'baseline': roi},
    landcover_df=scheme_df,
    aoi_geometry=aoi,
    period_order=mt_composites['period_order'],
    reference_period='baseline',
    class_field=CLASS_ID_PROP,
    use_class_ids=True,
    min_samples=5,
)
print('Detected mode:', train_loaded['mode'])
print('Validation summary:', train_loaded['validation_summary'])

# Live GEE separability check (Model Transfer skips historical)
sep_res = sample_quality.run_multitemporal_separability(
    composites_by_period=mt_composites,
    training_by_period=train_loaded['training_by_period'],
    class_property=CLASS_ID_PROP,
    region=geom,
    class_name_property=CLASS_NAME_PROP,
    scale=30,
    max_pixels_per_class=5000,
)
print('Analyzed periods:', sep_res['analyzed_periods'])
print('Skipped periods:', sep_res['skipped_periods'])
display(sep_res['per_period']['baseline']['summary_df'])
display(sep_res['per_period']['baseline']['separability_df'])

---
## Phase 3 Live EE Tests: Early Phase Predictor (`multitemporal.py`)
Verifies that predictors (`common_bands` + terrain + compatible spectral indices) are computed on **both** `historical` and `baseline` with identical band lists.

In [ ]:
from luma_ge.multitemporal import compute_multitemporal_predictors

predictor_config = {
    'individual_predictors': {
        'elevation': True,
        'slope': True,
        'aspect': False,
        'spectral_indices': ['NDVI', 'MNDWI', 'NDBI'],
    }
}

pred_res = compute_multitemporal_predictors(
    composites_by_period=mt_composites,
    aoi=geom,
    predictor_config=predictor_config,
    collections_by_period=mt_data,
    mode='model_transfer',
)
print('Compatible indices:', pred_res['compatible_indices'])
print('Skipped indices:', pred_res['skipped_indices'])
print('Common feature bands:', pred_res['common_feature_bands'])
for p, stack in pred_res['predictor_stacks'].items():
    print(f'Live EE bands for {p}:', stack.bandNames().getInfo())

---
## Phase 4 Live EE Tests: `MULTIPROBABILITY` Model Transfer (`classification.py`) & Accuracy (`accuracy.py`)

In [ ]:
from luma_ge.classification import FeatureExtraction, Generate_LULC

# Stratified split + pixel extraction now happen directly in FeatureExtraction:
# 1 training dataset -> single extraction on reference_period; N datasets -> batch extraction.
fe = FeatureExtraction()
fe_res = fe.extract_multitemporal_samples(
    predictor_stacks=pred_res['predictor_stacks'],
    training_splits=train_loaded['training_by_period'],
    class_property=CLASS_ID_PROP,
    reference_period='baseline',
    common_bands=pred_res['common_feature_bands'],
    scale=30,
    tile_scale=4,
    split_method='stratified',
    train_ratio=0.7,
    seed=42,
    period_order=pred_res['period_order'],
)
print('Feature extraction type:', fe_res['extraction_type'], '| Mode:', fe_res['mode_used'])
print('Extracted periods:', fe_res['extracted_periods'], '| Skipped:', fe_res['skipped_periods'])

lulc = Generate_LULC()
clf_res = lulc.classify_multitemporal(
    predictor_stacks=pred_res['predictor_stacks'],
    training_splits=fe_res['sampled_splits'],
    class_property=CLASS_ID_PROP,
    mode=fe_res['mode_used'],
    reference_period='baseline',
    classification_type='soft',
    soft_mode='multiprobability',
    ntrees=100,
    common_bands=pred_res['common_feature_bands'],
    period_order=pred_res['period_order'],
    class_names=scheme_df,
)
print('Mode used:', clf_res['mode_used'])
print('Class IDs:', clf_res['class_ids'])
print('Probability bands:', clf_res['prob_bands'])
print('Baseline model eval OA:', clf_res['model_evaluation']['baseline']['overall_accuracy'])
print('Baseline model eval Kappa:', clf_res['model_evaluation']['baseline']['kappa'])
display(clf_res['feature_importance']['baseline'].head(10))

for p, m in clf_res['maps'].items():
    print(f'Classified map bands ({p}):', m.bandNames().getInfo())

Map4 = geemap.Map()
vis_lulc = {
    'bands': ['classification'],
    'min': int(scheme_df['ID'].min()),
    'max': int(scheme_df['ID'].max()),
    'palette': scheme_df.sort_values('ID')['Color Palette'].tolist(),
}
Map4.addLayer(clf_res['maps']['historical'].clip(geom), vis_lulc, '2010 Historical LULC (Transferred)')
Map4.addLayer(clf_res['maps']['baseline'].clip(geom), vis_lulc, '2025 Baseline LULC')
Map4.addLayer(clf_res['maps']['historical'].select('confidence').clip(geom), {'min': 25, 'max': 100}, '2010 Transfer Confidence (%)', False)
Map4.centerObject(aoi, 12)
Map4

In [ ]:
from luma_ge.accuracy import thematic_accuracy

# Auto-detect ground truth class property on the validation FeatureCollection
val_props = validation.first().propertyNames().getInfo()
print('Validation asset properties:', val_props)
val_class_prop = next(
    (c for c in ('ID_Kelas', 'class_id', 'Is labeled', 'kelas', 'ID') if c in val_props),
    val_props[0],
)
print('Using validation class property:', val_class_prop)

taa = thematic_accuracy()
acc_res = taa.run_multitemporal_assessment(
    maps_by_period=clf_res['maps'],
    validation_by_period={'baseline': validation, 'historical': None},
    class_property=val_class_prop,
    scale=30,
    required_periods=['baseline'],
    period_order=clf_res['period_order'],
)
print('Assessed periods:', acc_res['assessed_periods'])
print('Skipped periods:', acc_res['skipped_periods'])
ok_base, base_metrics = acc_res['per_period']['baseline']
print('Baseline Thematic Accuracy Summary:', taa.format_accuracy_summary(base_metrics) if ok_base else base_metrics)

---
## Phase 5 Live EE Tests: `MultitemporalAnalysis` End-to-End Orchestrator
Runs the complete pipeline via `PeriodConfig` + `MultitemporalAnalysis.run_pipeline()` (`acquire_imagery` → `load_training_data` → `check_separability` → `compute_predictors` → `extract_features` → `run_classification` → `assess_accuracy`).

In [ ]:
from luma_ge.multitemporal import PeriodConfig, MultitemporalAnalysis

mt = MultitemporalAnalysis(
    aoi=aoi,
    periods=[
        PeriodConfig(label='baseline', start_date=2025, end_date=2025, sensor='L8_SR', cloud_cover=30),
        PeriodConfig(label='historical', start_date=2010, end_date=2010, sensor='L5_SR', cloud_cover=60),
    ],
    reference_period='baseline',
    harmonize_landsat=True,
    apply_brdf=True,
    apply_topo_correction=False,
    histogram_match=True,
    scale=30,
)

mt_result = mt.run_pipeline(
    training_sources={'historical': None, 'baseline': roi},
    landcover_df=scheme_df,
    predictor_config={
        'individual_predictors': {
            'elevation': True,
            'slope': True,
            'aspect': False,
            'spectral_indices': ['NDVI', 'MNDWI', 'NDBI'],
        }
    },
    validation_by_period={'baseline': validation, 'historical': None},
    class_field='ID_Kelas',
    class_name_property='KelasTupla',
    use_class_ids=True,
    min_samples=5,
    run_separability=False,  #already verified in Phase 2; set True to include
    classification_type='soft',
    soft_mode='multiprobability',
    ntrees=100,
    verbose=True,
)

print('Chronological period_order:', mt_result['period_order'])
print('Mode detected:', mt_result['mode'])
print('Feature extraction type:', mt_result['feature_extraction']['extraction_type'])
print('Extracted periods:', mt_result['feature_extraction']['extracted_periods'])
print('Common predictor bands:', mt_result['predictors']['common_feature_bands'])
print('Baseline classified bands:', mt_result['classification']['maps']['baseline'].bandNames().getInfo())
ok_p5, base_acc_p5 = mt_result['accuracy']['per_period']['baseline']
print('Baseline accuracy (OA, Kappa):', (base_acc_p5['overall_accuracy'], base_acc_p5['kappa']) if ok_p5 else base_acc_p5)

---
## Phase 6 Live EE Tests: `ConsistencyAnalyzer` (`post_classification.py`)
Tests:
1. **6A**: `validate_transition_matrix()` across Kambera LULC classes `1..9` (marking urban-to-forest reversions like `7 -> 1, 2` as forbidden).
2. **6B**: `cross_tabulate()` (server-side transition counts, area, and spatial inconsistency mask).
3. **6C**: `correct_inconsistencies()` ? Primary HMM/Viterbi constrained MAP over `prob_<Class_Name>` bands vs. Secondary rule-based reclassification.

In [ ]:
from luma_ge.post_classification import ConsistencyAnalyzer

ca = ConsistencyAnalyzer()
class_ids = sorted(int(c) for c in scheme_df['ID'].tolist())

# Example ecological & urban permanence transition rules:
# Build full binary/weighted matrix: built-up (7) cannot revert to primary/mangrove forest (1, 2) or cropland (5, 6)
forbidden = {
    7: {1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 5: 0.0, 6: 0.0},
    8: {1: 0.0},
}
matrix_dict = {
    fid: {tid: (1.0 if fid == tid else forbidden.get(fid, {}).get(tid, 0.8)) for tid in class_ids}
    for fid in class_ids
}

val_mat = ca.validate_transition_matrix(matrix_dict, class_ids=class_ids)
print('Matrix valid:', val_mat['valid'], '| Missing pairs:', val_mat['missing_pairs'])

past_map = clf_res['maps']['historical']
future_map = clf_res['maps']['baseline']
past_prob_map = clf_res.get('prob_bands', {}).get('historical')

# Step 6B: Cross-tabulate raw historical vs baseline
ct_raw = ca.cross_tabulate(
    past_map=past_map,
    future_map=future_map,
    consistency_matrix=val_mat,
    aoi=geom,
    scale=30,
    class_names=scheme_df,
    class_ids=class_ids,
)
print(f"Raw pixels: {ct_raw['total_pixels']} | Inconsistent: {ct_raw['inconsistent_pixels']}")
display(ct_raw['inconsistent_transitions'])

# Step 6C (Primary): HMM anchored MAP with 3x3 spatial smoothing
hmm_res = ca.correct_inconsistencies(
    past_map=past_map,
    future_map=future_map,
    consistency_matrix=val_mat,
    aoi=geom,
    method='hmm',
    anchor_reference=True,
    spatial_smoothing=True,
    spatial_weight=0.25,
    scale=30,
    class_ids=class_ids,
    class_names=scheme_df,
    prob_band_map=past_prob_map,
)
print('HMM method:', hmm_res['method_used'], '| Corrected px:', hmm_res['corrected_pixel_count'], '| Remaining inconsistent:', hmm_res['remaining_inconsistent_count'])

# Step 6C (Secondary fallback): Rule-based reclassification comparison
rule_res = ca.correct_inconsistencies(
    past_map=past_map.select('classification'),
    future_map=future_map,
    consistency_matrix=val_mat,
    aoi=geom,
    method='auto',
    rules={(7, 1): 3},
    scale=30,
    class_ids=class_ids,
)
print('Rule fallback method:', rule_res['method_used'], '| Corrected px:', rule_res['corrected_pixel_count'], '| Remaining inconsistent:', rule_res['remaining_inconsistent_count'])

Map6 = geemap.Map()
vis_lulc = {
    'bands': ['classification'],
    'min': int(scheme_df['ID'].min()),
    'max': int(scheme_df['ID'].max()),
    'palette': scheme_df.sort_values('ID')['Color Palette'].tolist(),
}
Map6.addLayer(past_map.select('classification').clip(geom), vis_lulc, '2010 Raw Historical')
Map6.addLayer(hmm_res['corrected_map'].clip(geom), vis_lulc, '2010 HMM Consistent')
Map6.addLayer(ct_raw['inconsistent_mask'].clip(geom), {'palette': ['#FF00FF']}, 'Detected Inconsistent Pixels')
Map6.centerObject(aoi, 12)
Map6